# Vídeo con YOLO: detección y seguimiento
Este notebook muestra cómo dar el salto desde OpenCV clásico a detección moderna con un modelo preentrenado tipo YOLO aplicado sobre vídeo. La idea es detectar objetos por frame y, a partir de ahí, construir seguimiento y análisis más útiles.

## Qué vamos a hacer
1. Cargar un vídeo o una captura de cámara.
2. Aplicar un modelo YOLO preentrenado para detectar objetos en cada frame.
3. Dibujar las cajas y etiquetas sobre el vídeo.
4. Probar una forma básica de seguimiento.
5. Ver cómo reutilizar la misma idea en tiempo real con la webcam.

Nota: aquí usaremos la librería `ultralytics`. Aunque en algunos materiales aparezca el nombre YOLOv11, el flujo práctico es muy parecido al de otros modelos recientes de la familia YOLO.

## Requisitos
- Un entorno con `python`, `opencv-python` y `ultralytics` instalados.
- Si quieres abrir ventanas con `cv2.imshow`, necesitas ejecutarlo en local con acceso gráfico.
- Para la cámara en vivo, el equipo debe tener webcam accesible desde OpenCV.

In [ ]:
# Instalación orientativa si hace falta
# !pip install -U ultralytics opencv-python

In [ ]:
# Librerías
from pathlib import Path
import os

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display, Markdown

try:
    from ultralytics import YOLO
except Exception as e:
    print("Ultralytics no disponible:", e)

print("Directorio actual:", os.getcwd())

## Selección del vídeo
Puedes usar un archivo de vídeo ya existente o subir uno si estás en Colab.

In [ ]:
VIDEO_PATH = Path("video.mp4")

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded)))
except Exception:
    pass

print(f"Vídeo seleccionado: {VIDEO_PATH}")
print(f"¿Existe?: {VIDEO_PATH.exists()}")

In [ ]:
# Vista previa rápida
if VIDEO_PATH.exists():
    display(Video(str(VIDEO_PATH), embed=True))
else:
    print("No se ha encontrado el vídeo. Puedes seguir el notebook y aportar un archivo después.")

## Carga del modelo YOLO preentrenado
Usamos un modelo pequeño para que la inferencia sea más ligera. Si tu equipo tiene más capacidad, puedes probar variantes más grandes.

In [ ]:
model = YOLO("yolov8n.pt")
model

## 1. Detección sobre un frame de ejemplo
Antes de procesar todo el vídeo, conviene comprobar cómo responde el detector sobre un fotograma.

In [ ]:
def read_first_frame(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el vídeo: {video_path}")
    ret, frame = cap.read()
    cap.release()
    if not ret or frame is None:
        raise ValueError("No se pudo leer el primer frame del vídeo.")
    return frame

if VIDEO_PATH.exists():
    frame = read_first_frame(VIDEO_PATH)
    results = model.predict(source=frame, verbose=False)
    annotated = results[0].plot()

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Detección YOLO sobre el primer frame")
    plt.show()
else:
    print("No hay vídeo cargado.")

## 2. Detección YOLO sobre todo el vídeo
Esta celda recorre el vídeo frame a frame, ejecuta inferencia y muestra el resultado en una ventana local.

In [ ]:
# Detección en vídeo con visualización local
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = model.predict(source=frame, verbose=False)
    annotated = results[0].plot()
    cv2.imshow("YOLO deteccion", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 3. Guardar el vídeo anotado
En muchos casos interesa generar un nuevo archivo con las detecciones dibujadas para revisarlo después.

In [ ]:
# Guardar una copia anotada del vídeo
output_video = "video_yolo_anotado.mp4"

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 20.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = model.predict(source=frame, verbose=False)
    annotated = results[0].plot()
    writer.write(annotated)

cap.release()
writer.release()
print(f"Vídeo guardado en: {output_video}")

In [ ]:
# Mostrar el vídeo anotado si existe
if Path(output_video).exists():
    display(Video(output_video, embed=True))

## 4. Seguimiento básico
YOLO detecta objetos en cada frame. Para mantener identidades a lo largo del tiempo, necesitamos tracking. La librería Ultralytics ya incorpora un método `track()` que simplifica mucho este paso.

In [ ]:
# Tracking con YOLO sobre vídeo
# persist=True intenta mantener identidades entre frames
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

results = model.track(source=str(VIDEO_PATH), show=True, persist=True)
print("Tracking terminado.")

### Nota sobre la celda anterior
La forma más sencilla de usar tracking con Ultralytics es delegar el procesamiento completo a `model.track(source=...)`. Si quieres más control frame a frame, ya necesitas un pipeline algo más largo y específico.

## 5. Cámara en vivo con YOLO
La misma idea puede aplicarse a la webcam. Esto es especialmente útil para demos en clase, siempre que el equipo tenga suficiente rendimiento.

In [ ]:
# Detección en vivo con la cámara
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = model.predict(source=frame, verbose=False)
    annotated = results[0].plot()
    cv2.imshow("YOLO en vivo", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## Qué aporta YOLO frente a OpenCV clásico
- Detecta objetos semánticos: persona, coche, perro, bicicleta, etc.
- Funciona bien en escenas complejas donde una simple diferencia de frames no basta.
- Se puede combinar con tracking, conteo, zonas de interés y análisis de eventos.
- Es una puerta natural hacia aplicaciones reales de vídeo inteligente.

In [ ]:
Markdown('''
**Actividad sugerida**

1. Prueba el detector YOLO sobre un vídeo corto con personas o vehículos.
2. Compara el resultado con la detección clásica por movimiento de OpenCV.
3. Guarda el vídeo anotado y revisa en qué escenas el modelo funciona mejor o peor.
4. Si tu equipo lo permite, ejecuta la versión en vivo con la cámara.
5. Como ampliación, usa un modelo de pose o segmentación en vez del detector general.
''')